# AudioEditingCode - Google Colab

This notebook runs the [AudioEditingCode](https://github.com/BF667-IDLE/AudioEditingCode) project on Google Colab with Python 3.10+ and modern PyTorch.

## Features
- Text-based audio editing via DDPM inversion
- Principal Component (PC) extraction and drift application
- SDEdit-based editing

**Make sure you're running on a GPU runtime** (Runtime > Change runtime type > GPU).

## 1. Setup

In [ ]:
# Check Python and GPU
import sys
print(f"Python version: {sys.version}")

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone the repository
!git clone https://github.com/BF667-IDLE/AudioEditingCode.git
%cd AudioEditingCode

In [ ]:
# Install dependencies
!pip install -r requirements.txt

In [ ]:
# Add code directory to path so imports work
import sys
import os

code_dir = os.path.join(os.getcwd(), 'code')
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)

# Also add the audioldm subdirectory
audioldm_dir = os.path.join(code_dir, 'audioldm')
if audioldm_dir not in sys.path:
    sys.path.insert(0, audioldm_dir)

print(f"Working directory: {os.getcwd()}")
print(f"Code directory: {code_dir}")

## 2. Upload Audio

Upload an audio file (WAV format recommended) to the Colab workspace.

In [ ]:
# Option 1: Upload from local machine
from google.colab import files
import os

os.makedirs('input_audio', exist_ok=True)
uploaded = files.upload()
for filename in uploaded.keys():
    os.rename(filename, os.path.join('input_audio', filename))
    print(f"Uploaded: {filename}")

# Set the audio path
AUDIO_PATH = os.path.join('input_audio', list(uploaded.keys())[0])
print(f"Audio path: {AUDIO_PATH}")

In [ ]:
# Option 2: Use a sample audio (uncomment to use)
# !wget -O input_audio/sample.wav "https://example.com/sample.wav"
# AUDIO_PATH = "input_audio/sample.wav"
# print(f"Audio path: {AUDIO_PATH}")

## 3. Text-Based Audio Editing (DDPM Inversion)

Edit an audio clip by inverting it to the latent space and regenerating with a new text prompt.

In [ ]:
# Text-based audio editing
!python code/main_run.py \
    --init_aud "{AUDIO_PATH}" \
    --model_id "cvssp/audioldm2-music" \
    --source_prompt "" \
    --target_prompt "a jazz version with saxophone" \
    --target_neg_prompt "low quality, distorted" \
    --cfg_src 3 \
    --cfg_tar 12 \
    --num_diffusion_steps 200 \
    --tstart 100 \
    --results_path "results" \
    --wandb_disable

In [ ]:
# Play the result
import glob
import IPython.display as ipd

result_files = sorted(glob.glob('results/**/*.wav', recursive=True))
if result_files:
    print(f"Generated files: {result_files}")
    for f in result_files:
        print(f"\nPlaying: {f}")
        ipd.display(ipd.Audio(f))
else:
    print("No results found. Check the output above for errors.")

## 4. PC Extraction

Extract Principal Components from an audio signal's inversion trajectory.

In [ ]:
# Extract PCs
!python code/main_pc_extract_inv.py \
    --init_aud "{AUDIO_PATH}" \
    --model_id "cvssp/audioldm2-music" \
    --source_prompt "" \
    --target_neg_prompt "" \
    --num_diffusion_steps 200 \
    --cfg_tar 3 \
    --n_evs 3 \
    --iters 50 \
    --results_path "pc_extractions" \
    --wandb_disable

## 5. Apply PC Drift

Apply the extracted Principal Components to edit the audio in a specific direction.

In [ ]:
# Find the extraction checkpoint
import glob
extraction_files = sorted(glob.glob('pc_extractions/**/*.pt', recursive=True))
if extraction_files:
    EXTRACTION_PATH = extraction_files[-1]
    print(f"Using extraction: {EXTRACTION_PATH}")
else:
    raise FileNotFoundError("No extraction checkpoint found. Run PC extraction first.")

In [ ]:
# Apply PC drift
!python code/main_pc_apply_drift.py \
    --extraction_path "{EXTRACTION_PATH}" \
    --drift_start 200 \
    --drift_end -1 \
    --amount 2.0 \
    --evs 1 \
    --wandb_disable

In [ ]:
# Play the PC drift results
import glob
import IPython.display as ipd

drift_files = sorted(glob.glob('**/*driftgens/*.wav', recursive=True))
if drift_files:
    for f in drift_files:
        print(f"Playing: {f}")
        ipd.display(ipd.Audio(f))
else:
    print("No drift results found. Check the output above for errors.")

## 6. SDEdit

Simple SDEdit-based editing with a new prompt.

In [ ]:
# SDEdit
!python code/main_run_sdedit.py \
    --init_aud "{AUDIO_PATH}" \
    --model_id "cvssp/audioldm2-music" \
    --target_prompt "a calm piano version" \
    --target_neg_prompt "" \
    --cfg_tar 12 \
    --num_diffusion_steps 200 \
    --tstart 100 \
    --results_path "sdedit_results" \
    --wandb_disable

In [ ]:
# Play SDEdit results
import glob
import IPython.display as ipd

sdedit_files = sorted(glob.glob('sdedit_results/**/*.wav', recursive=True))
if sdedit_files:
    for f in sdedit_files:
        print(f"Playing: {f}")
        ipd.display(ipd.Audio(f))
else:
    print("No SDEdit results found. Check the output above for errors.")

## 7. Download Results

Download all generated audio files.

In [ ]:
# Download all results as a zip
!zip -r results.zip results/ pc_extractions/ sdedit_results/ 2>/dev/null || true
from google.colab import files
files.download('results.zip')